# 02 — Qualité et hallucinations données

Hallucination = valeur **impossible ou contradictoire** (pas une invention de LLM). Exemples pseudonymisés uniquement (`PatientID` / `SejourID`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
from src.data.paths import ProjectPaths, get_project_root
from src.data.load import load_csv
from src.data.quality import (
    recalculate_duree_sejour, flag_signes_vitaux, harmonize_biologie, filter_objets_connectes_bons
)

paths = ProjectPaths(root=get_project_root())
sejours = recalculate_duree_sejour(load_csv(paths, "sejours.csv", from_raw=False))
print("durées incohérentes:", int(sejours["flag_duree_incoherente"].sum()))
sejours.loc[sejours["flag_duree_incoherente"], ["SejourID", "DureeSejour_brute", "DureeSejour"]].head()

durées incohérentes: 42


,SejourID,DureeSejour_brute,DureeSejour
49,SEJ-000050,-1,2
50,SEJ-000051,-3,3
76,SEJ-000077,0,2
84,SEJ-000085,99,6
100,SEJ-000101,0,5


In [3]:
sv = flag_signes_vitaux(load_csv(paths, "signes_vitaux.csv", from_raw=False))
flags = [c for c in sv.columns if c.startswith("flag_")]
sv[flags].mean()

flag_FrequenceCardiaque_aberrant       0.095922
flag_TensionSystolique_aberrant        0.051172
flag_TensionDiastolique_aberrant       0.059038
flag_Temperature_aberrant              0.067146
flag_FrequenceRespiratoire_aberrant    0.081113
flag_SpO2_aberrant                     0.070958
dtype: float64

In [4]:
bio = harmonize_biologie(load_csv(paths, "biologies.csv", from_raw=False))
print("bio aberrante:", float(bio["flag_biologie_aberrante"].mean()))
oc = load_csv(paths, "objets_connectes.csv", from_raw=False)
print("objets Bon:", len(filter_objets_connectes_bons(oc)), "/", len(oc))

bio aberrante: 0.45471647406611126
objets Bon: 5125 / 6297


In [5]:
assert "flag_duree_incoherente" in sejours.columns
print("OK qualité")

OK qualité
